# SensorVision: Industrial Fault Detection with 1D CNN

**arXiv ref:** *Multimodal Bearing Fault Classification with 1D CNN* — arxiv:2502.17524, Feb 2025

**MRMS-Net:** *Scalable Multi-Scale Convolutional Framework for Time Series* — arxiv:2603.19315, Mar 2026

This notebook demonstrates multi-scale 1D CNN for classifying industrial sensor faults.


In [ ]:
from __future__ import annotations
import sys, os; sys.path.insert(0, os.path.abspath('.'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import make_synthetic, create_dataloaders, FAULT_CLASSES
from src.model import MultiScale1DCNN
from src.core import compute_sensor_metrics
from src.visualizations import plot_signals, plot_confusion_matrix


In [ ]:
# Generate synthetic sensor data
data = make_synthetic(n_per_class=100, seed=42)
print(f"Signals: {data['n_samples']}")
print(f"Fault types: {data['class_names']}")
print(f"Signal length: {data['signal_length']}")


# Visualize signal types
fig = plot_signals(data['signals'], data['labels'], FAULT_CLASSES, n=5)
plt.show()


In [ ]:
# Build multi-scale 1D CNN
model = MultiScale1DCNN(num_classes=data['num_classes'], signal_length=data['signal_length'])
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
print("Architecture: 3 parallel branches with kernel sizes 3, 15, 63 → concat → FC classifier")


In [ ]:
# Forward pass
with torch.no_grad():
    out = model(data['signals'][:8])
    preds = out.argmax(dim=1)
print(f"Output: {out.shape}")
print(f"Predictions: {preds}")


In [ ]:
# Classification report
metrics = compute_sensor_metrics(preds.numpy(), data['labels'][:8].numpy(), FAULT_CLASSES)
print(f"Accuracy: {metrics['accuracy']:.4f}")
for cls, m in metrics['per_class'].items():
    print(f"  {cls:15s} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}")


In [ ]:
print('SensorVision notebook complete. Multi-scale 1D CNNs capture fault patterns at different frequency bands.')
